In [1]:
import requests
import json
import pandas as pd
from pathlib import Path

In [2]:
latitude = 40.6413
longitude = -73.7781

In [3]:
start_date = "2025-01-01"
end_date = "2025-01-31"

In [4]:
hourly_variables = [
    "temperature_2m",
    "relative_humidity_2m",
    "precipitation",
    "rain",
    "snowfall",
    "weather_code",
    "pressure_msl",
    "cloud_cover",
    "wind_speed_10m",
    "wind_direction_10m",
    "wind_gusts_10m"
]

In [5]:
url = "https://archive-api.open-meteo.com/v1/archive"

params = {
    "latitude": latitude,
    "longitude": longitude,
    "start_date": start_date,
    "end_date": end_date,
    "hourly": ",".join(hourly_variables),
    "timezone": "auto"
}

response = requests.get(url, params=params, timeout=30)
response.raise_for_status()

data = response.json()

In [6]:
data.keys()

dict_keys(['latitude', 'longitude', 'generationtime_ms', 'utc_offset_seconds', 'timezone', 'timezone_abbreviation', 'elevation', 'hourly_units', 'hourly'])

In [7]:
data["timezone"]

'America/New_York'

In [8]:
data["hourly"].keys()

dict_keys(['time', 'temperature_2m', 'relative_humidity_2m', 'precipitation', 'rain', 'snowfall', 'weather_code', 'pressure_msl', 'cloud_cover', 'wind_speed_10m', 'wind_direction_10m', 'wind_gusts_10m'])

In [9]:
raw_dir = Path("../data/raw/weather/JFK")
raw_dir.mkdir(parents=True, exist_ok=True)

json_path = raw_dir / "2025_01.json"

with open(json_path, "w", encoding="utf-8") as f:
    json.dump(data, f, ensure_ascii=False, indent=2)

In [10]:
weather_df = pd.DataFrame(data["hourly"])

In [11]:
weather_df.shape

(744, 12)

In [12]:
weather_df.head()

,time,temperature_2m,relative_humidity_2m,precipitation,rain,snowfall,weather_code,pressure_msl,cloud_cover,wind_speed_10m,wind_direction_10m,wind_gusts_10m
0,2025-01-01T00:00,9.4,94,4.2,4.2,0.0,63,1000.7,100,15.1,102,37.1
1,2025-01-01T01:00,9.9,97,3.0,3.0,0.0,63,998.9,100,15.8,163,30.2
2,2025-01-01T02:00,9.8,96,0.0,0.0,0.0,1,997.9,34,17.6,180,36.4
3,2025-01-01T03:00,9.1,98,0.0,0.0,0.0,3,997.5,100,12.5,187,33.8
4,2025-01-01T04:00,8.9,98,0.0,0.0,0.0,3,997.2,100,9.7,223,24.1


In [13]:
weather_df.dtypes

time                        str
temperature_2m          float64
relative_humidity_2m      int64
precipitation           float64
rain                    float64
snowfall                float64
weather_code              int64
pressure_msl            float64
cloud_cover               int64
wind_speed_10m          float64
wind_direction_10m        int64
wind_gusts_10m          float64
dtype: object

In [14]:
weather_df["time"] = pd.to_datetime(weather_df["time"])

In [15]:
weather_df.dtypes

time                    datetime64[us]
temperature_2m                 float64
relative_humidity_2m             int64
precipitation                  float64
rain                           float64
snowfall                       float64
weather_code                     int64
pressure_msl                   float64
cloud_cover                      int64
wind_speed_10m                 float64
wind_direction_10m               int64
wind_gusts_10m                 float64
dtype: object

In [16]:
weather_df.isna().sum()

time                    0
temperature_2m          0
relative_humidity_2m    0
precipitation           0
rain                    0
snowfall                0
weather_code            0
pressure_msl            0
cloud_cover             0
wind_speed_10m          0
wind_direction_10m      0
wind_gusts_10m          0
dtype: int64

In [17]:
weather_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 744 entries, 0 to 743
Data columns (total 12 columns):
 #   Column                Non-Null Count  Dtype         
---  ------                --------------  -----         
 0   time                  744 non-null    datetime64[us]
 1   temperature_2m        744 non-null    float64       
 2   relative_humidity_2m  744 non-null    int64         
 3   precipitation         744 non-null    float64       
 4   rain                  744 non-null    float64       
 5   snowfall              744 non-null    float64       
 6   weather_code          744 non-null    int64         
 7   pressure_msl          744 non-null    float64       
 8   cloud_cover           744 non-null    int64         
 9   wind_speed_10m        744 non-null    float64       
 10  wind_direction_10m    744 non-null    int64         
 11  wind_gusts_10m        744 non-null    float64       
dtypes: datetime64[us](1), float64(7), int64(4)
memory usage: 69.9 KB


In [18]:
weather_df["time"].min(), weather_df["time"].max()

(Timestamp('2025-01-01 00:00:00'), Timestamp('2025-01-31 23:00:00'))

In [19]:
weather_df["time"].nunique()

744

In [20]:
weather_df["time"].duplicated().sum()

np.int64(0)

In [21]:
weather_df.describe()

,time,temperature_2m,relative_humidity_2m,precipitation,rain,snowfall,weather_code,pressure_msl,cloud_cover,wind_speed_10m,wind_direction_10m,wind_gusts_10m
count,744,744.000000,744.000000,744.000000,744.000000,744.000000,744.000000,744.000000,744.000000,744.000000,744.000000,744.000000
mean,2025-01-16 11:30:00,-1.374328,60.405914,0.038710,0.024059,0.010255,6.146505,1015.222446,50.233871,14.212634,269.883065,33.602419
min,2025-01-01 00:00:00,-13.800000,36.000000,0.000000,0.000000,0.000000,0.000000,996.900000,0.000000,0.900000,1.000000,2.900000
25%,2025-01-08 17:45:00,-3.625000,50.000000,0.000000,0.000000,0.000000,0.000000,1009.200000,4.000000,9.500000,253.000000,22.000000
50%,2025-01-16 11:30:00,-1.500000,59.000000,0.000000,0.000000,0.000000,1.000000,1015.100000,44.000000,13.550000,281.000000,31.700000
75%,2025-01-24 05:15:00,1.700000,67.000000,0.000000,0.000000,0.000000,3.000000,1020.500000,99.000000,18.800000,304.000000,43.900000
max,2025-01-31 23:00:00,10.600000,98.000000,4.200000,4.200000,1.400000,75.000000,1036.400000,100.000000,36.100000,360.000000,82.100000
std,NaN,4.720705,13.920047,0.248292,0.219504,0.083334,16.848667,8.633659,41.931292,6.311241,58.613211,15.250510


In [22]:
weather_df["date"] = weather_df["time"].dt.date

daily_weather = (
    weather_df.groupby("date")
    .agg(
        avg_temperature=("temperature_2m", "mean"),
        precipitation_sum=("precipitation", "sum"),
        snowfall_sum=("snowfall", "sum"),
        max_wind_speed=("wind_speed_10m", "max"),
        max_wind_gust=("wind_gusts_10m", "max"),
        avg_pressure=("pressure_msl", "mean")
    )
    .reset_index()
)

daily_weather.head()

,date,avg_temperature,precipitation_sum,snowfall_sum,max_wind_speed,max_wind_gust,avg_pressure
0,2025-01-01,7.841667,7.2,0.0,25.8,70.6,1000.845833
1,2025-01-02,2.875000,0.0,0.0,26.3,67.7,1012.745833
2,2025-01-03,0.616667,0.0,0.0,18.7,45.0,1012.500000
3,2025-01-04,-1.245833,0.0,0.0,27.9,67.3,1011.254167
4,2025-01-05,-1.912500,0.0,0.0,22.1,54.0,1016.725000


In [23]:
weather_df["airport_code"] = "JFK"

In [24]:
timezone = data["timezone"]
timezone

'America/New_York'